In [1]:
%gui asyncio

In [2]:
import numpy as np
import scipy
import tifffile
from tqdm import tqdm
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import pairwise_distances
import pygpcca

import numba

In [3]:
vd1 = {
    "path": "/home/kushal/lab/datasets/jason/M19/D05/M19D5_MC.tif",
    "rects": [(162, 268, 178, 288), (304, 410, 118, 228), (345, 451, 297, 407)],
    "results": list()
}

vd2 = {
    "path": "/home/kushal/lab/datasets/jason/61/D05/61D5_MC_mmap.tif",
    "rects": [(132, 290, 92, 264), (296, 461, 197, 414)],
    "results": list()
}

vd3 = {
    "path": "/home/kushal/lab/datasets/jason/885/D05/885D5_MC.tif",
    "rects": [(155, 307, 166, 326), (231, 383, 99, 259), (126, 278, 91, 251), (47, 210, 197, 357), (179, 328, 270, 398)],
    "results": list()
}

vids = [vd1, vd2, vd3]

In [4]:
@numba.guvectorize([(numba.float32[:], numba.float32[:])], "(n)->(n)")
def rolling_percentile(trace: np.ndarray, out: np.ndarray):
    for i in range(0, trace.size - 1):
        out[i] = np.percentile(trace[i:i + 30], 5)


@numba.njit(parallel=True)
def rolling_percentile_movie(m, output):
    t, row, col = m.shape
    for i in numba.prange(row):
        for j in range(col):
            rolling_percentile(m[:, i, j], output[:, i, j])

In [5]:
for vid in vids:
    movie_raw = tifffile.imread(vid["path"])[:10_000]
    rects = vid["rects"]
    print(f"movie: {Path(vid['path']).stem}")
    
    for rect_ix, rect in tqdm(enumerate(rects), total=len(rects)):
        print(f"rect {rect_ix} / {len(rects)}")
        
        c0, c1, r0, r1 = rect
        
        sub_raw = movie_raw[:, r0:r1:2, c0:c1:2].astype(np.float32).copy()

        sub = np.zeros(sub_raw.shape, dtype=np.float32)

        print("rolling percentile filter")
        rolling_percentile_movie(sub_raw, sub)

        Y = sub.reshape(sub.shape[0], sub.shape[1] * sub.shape[2]).T.copy()

        print("computing B")
        B = pairwise_distances(Y, Y, n_jobs=20, metric="cosine")

        d = 1.0 / np.sum(B.astype(np.float64), axis=0)

        print("computing A")
        A = np.diag(d) @ B

        del Y
        del sub
        del sub_raw
        
        # results = dict()

        # results["B"] = B
        # results["A"] = A
        
        gp = pygpcca.GPCCA(A, z="LM", method="brandts")

        print("computing min chi")
        mc = gp.minChi(10, 30)

        for n in tqdm(range(10, 25)):
            print(f"clustering with n = {n}")
            gp.optimize(n)

            gpcca_path = Path("./gpcca_results").joinpath(f"{Path(vid['path']).stem}_rect-{rect_ix}_clusters-{n}.pickle")
            pickle.dump(gp, open(gpcca_path, "wb"))

movie: M19D5_MC


  0%|                                                                                   | 0/3 [00:00<?, ?it/s]

rect 0 / 3
rolling percentile filter
computing B
computing A
computing min chi



  0%|                                                                                  | 0/15 [00:00<?, ?it/s]

clustering with n = 10



  7%|████▉                                                                     | 1/15 [00:03<00:48,  3.47s/it]

... (316 outputs left out)

clustering with n = 23



 93%|████████████████████████████████████████████████████████████████████▏    | 14/15 [06:18<00:49, 49.11s/it]

clustering with n = 24



100%|██████████████████████████████████████████████████████████████████████████| 5/5 [50:58<00:00, 611.75s/it]
